# Set up
Se asume que el lector tiene minimos conocimientos de Python y como funciona. Antes de iniciar ningun proceso, se instalan las librerias necesarias.

In [ ]:
%pip install mujoco numpy matplotlib jupyter pandas

# Estructura
La manera de gestionar información se divide en dos grandes categorías:
- **mjModel**: Acá se almacena toda la información que es estática, como la estructura del robot, los pesos de cada parte, las relaciones, etc.
- **mjData**: Acá va a estar guardado el estado en tiempo real, las velocidades, posiciones, comandos, etc. que definen como está el robot en la simulación.


## Validación
El siguiente script está diseñado para evaluar secuencialmente el sobreimpulso (Step Response), el seguimiento dinámico y la estabilidad del componente integrador (kp) bajo múltiples frecuencias, para encontrar una respuesta satisfactoria de los motores.

In [ ]:
import mujoco
import numpy as np
import matplotlib.pyplot as plt

# Cargar el modelo
xml_path = "..\Cuerpo\DUM4.xml" #Ruta relativa a donde está almacenado el modelo
model = mujoco.MjModel.from_xml_path(xml_path) #Informacion estática
data = mujoco.MjData(model) #Informacion dinámica

# Simulacion
duration = 10.0 # Segundos
framerate = 60 
steps_per_frame = int((1.0/framerate)/model.opt.timestep)
total_frames = int(duration * framerate)

# Lista de todos los actuadores para probarlos
actuators_to_test = ["act_BaseHip","act_LeftShoulderArm", "act_Neck", "act_HipBody", "act_HeadBase", "act_HeadRot", "act_RightShoulderArm", "act_LeftForearm", "act_RightForearm", "act_LeftWrist", "act_RightWrist", "act_LeftLever_Slider", "act_RightLever_Slider"]

# Se cargan los limites de los controladores
ctrl_indices = {}
for name in actuators_to_test:
    actuator_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, name)
    if actuator_id == -1: # Si no fuese valido alguno:
        raise ValueError(f"Error: Actuador {name} no es válido")
    ctrl_indices[name] = actuator_id # Guardar el id para el nombre del actuador

# Buscar joint del actuador (para obtener su qpos)
qpos_indices = {}
for name in actuators_to_test:
    actuator_id = ctrl_indices[name]
    joint_id = model.actuator_trnid[actuator_id,0]
    qpos_idx = model.jnt_qposadr[joint_id]
    qpos_indices[name] = qpos_idx

# Crear log y telemetría de manera limpia
time_log = np.zeros(total_frames)
telemetry = {name: {'ctrl': np.zeros(total_frames), 'qpos': np.zeros(total_frames)} for name in actuators_to_test}
mujoco.mj_resetData(model, data)

# Inicio de la simulacion
for i in range(total_frames):
    time = data.time
    # Fase 1: Amortiguamiento (Kp/Kd)
    # Escalon de 0.4 rad al torso:
    target_hip = 0.4 if time > 0.5 else 0.0 #luego de medio segundo, cambio repentino de posicion
    
    #Fase 2: Limite y carga
    #Movimiento Sinusoidal rapido:
    if time > 3.0:
        target_shoulder = 0.08 * np.sin(2.0 * np.pi * 1.5 * (time - 3.0)) 
    else:
        target_shoulder = 0.0
        
    # Fase 3: Integrador
    # Suma de señal sinusoidal (simil ruido orgánico) en el cuello:
    if time > 7.0:
        target_neck = 0.3 * np.sin(time * 2.0) + 0.1 * np.sin(time *15.0)
    else: 
        target_neck = 0.0
        
    # Se envia siempre el valor, cuando el tiempo sea correcto se actualizara en simulacion:
    data.ctrl[ctrl_indices["act_HipBody"]] = target_hip
    data.ctrl[ctrl_indices["act_LeftShoulderArm"]] = target_shoulder
    data.ctrl[ctrl_indices["act_Neck"]] = target_neck
    
    # Registrar datos
    time_log[i] = time
    for name in actuators_to_test:
        telemetry[name]['ctrl'][i] = data.ctrl[ctrl_indices[name]]
        telemetry[name]['qpos'][i] = data.qpos[qpos_indices[name]]
        
    # Avanzar al siguiente step:
    for _ in range(steps_per_frame):
        mujoco.mj_step(model, data)



Cuando termine la prueba, se pueden graficar los resultados que se obtuvieron:

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle('Análisis de Telemetría Dinámica', fontsize=16)

for ax, name in zip(axs, actuators_to_test):
    ax.plot(time_log, telemetry[name]['ctrl'], 'r--', label='Comando Objetivo (Target)')
    ax.plot(time_log, telemetry[name]['qpos'], 'b-', label='Posición Real (qpos)')
    ax.set_ylabel('Posición (rad)')
    ax.set_title(f'Actuador: {name}')
    ax.legend()
    ax.grid(True)

axs[-1].set_xlabel('Tiempo (s)')
plt.tight_layout()
plt.show()

Para validar todos los joints y sus parámetros, se utiliza un script más integrador

In [ ]:
# Lista de todos los actuadores para probarlos
actuators_to_test = ["act_BaseHip","act_HipBody", "act_Neck",  "act_HeadBase", "act_HeadRot", "act_LeftShoulderArm", "act_LeftForearm", "act_LeftWrist", "act_LeftLever_Slider", "act_RightShoulderArm", "act_RightForearm", "act_RightWrist", "act_RightLever_Slider"]

# Se cargan los limites de los controladores
ctrl_indices = {}
for name in actuators_to_test:
    actuator_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, name)
    if actuator_id == -1: # Si no fuese valido alguno:
        raise ValueError(f"Error: Actuador {name} no es válido")
    ctrl_indices[name] = actuator_id # Guardar el id para el nombre del actuador

# Buscar joint del actuador (para obtener su qpos)
qpos_indices = {}
for name in actuators_to_test:
    actuator_id = ctrl_indices[name]
    joint_id = model.actuator_trnid[actuator_id,0]
    qpos_idx = model.jnt_qposadr[joint_id]
    qpos_indices[name] = qpos_idx

# Crear log y telemetría de manera limpia
time_log = np.zeros(total_frames)
telemetry = {name: {'ctrl': np.zeros(total_frames), 'qpos': np.zeros(total_frames)} for name in actuators_to_test}
mujoco.mj_resetData(model, data)
time_log = np.zeros(total_frames)
telemetry = {name: {
    'ctrl': np.zeros(total_frames), 
    'qpos': np.zeros(total_frames),
    'force': np.zeros(total_frames)
} for name in actuators_to_test}
# Inicio de la simulacion
for i in range(total_frames):
    time = data.time
    
    # --- GENERADOR DE SEÑALES DE PRUEBA EXHAUSTIVO (13 ACTUADORES) ---
    
    # Inicializamos todos los targets en 0 en cada paso
    targets = {name: 0.0 for name in actuators_to_test}

    # FASE 1: Estructura Central (Torso y Base) - 0s a 2s
    # Evalúa la capacidad de mover la masa principal del robot.
    if 0.0 <= time < 2.0:
        targets["act_BaseHip"] = 0.015 if time > 0.5 else 0.0
        targets["act_HipBody"] = 0.4 if time > 1.0 else 0.0

    # FASE 2: Complejo de Cabeza y Cuello - 2s a 4s
    # Evalúa la estabilidad de la cadena superior y el jitter.
    elif 2.0 <= time < 4.0:
        # Movimiento de búsqueda (Neck)
        targets["act_Neck"] = 0.5 * np.sin(2 * np.pi * 0.5 * (time - 2.0))
        # Respuesta rápida de orientación (HeadBase y Rot)
        targets["act_HeadBase"] = 0.3 if time > 3.0 else 0.0
        targets["act_HeadRot"] = 0.4 * np.sin(2 * np.pi * 2.0 * (time - 2.0))

    # FASE 3: Brazos - Hombros (Simetría) - 4s a 6s
    # Evalúa si ambos hombros responden igual bajo el límite restrictivo de 5 grados.
    elif 4.0 <= time < 6.0:
        signal = 0.08 * np.sin(2 * np.pi * 1.0 * (time - 4.0))
        targets["act_LeftShoulderArm"] = signal
        targets["act_RightShoulderArm"] = signal

    # FASE 4: Brazos - Antebrazos (Alcance) - 6s a 8s
    # Evalúa el torque en articulaciones de gran rango (1.57 rad).
    elif 6.0 <= time < 8.0:
        targets["act_LeftForearm"] = 1.2 if time > 6.5 else 0.0
        targets["act_RightForearm"] = 0.5 if time > 7.0 else 0.0

    # FASE 5: Terminales y Pinzas (Leva/Slider) - 8s a 10s
    # Prueba crítica de la restricción <equality>. 
    # El slider debe moverse y los dedos deben seguirlo por física, no por actuador.
    elif 8.0 <= time < 10.0:
        # Movimiento oscilatorio de las muñecas
        targets["act_LeftWrist"] = 1.0 * np.sin(2 * np.pi * 0.5 * (time - 8.0))
        targets["act_RightWrist"] = -1.0 * np.sin(2 * np.pi * 0.5 * (time - 8.0))
        # Acción de la pinza (Slider de la leva)
        # Se abre y cierra a 1Hz
        targets["act_LeftLever_Slider"] = 0.005 + 0.005 * np.sin(2 * np.pi * 1.0 * (time - 8.0))
        targets["act_RightLever_Slider"] = 0.005 + 0.005 * np.sin(2 * np.pi * 1.0 * (time - 8.0))

    # Aplicar todos los targets calculados al modelo
    for name in actuators_to_test:
        data.ctrl[ctrl_indices[name]] = targets[name]
    
    # Registrar datos
    time_log[i] = time
    for name in actuators_to_test:
        idx = ctrl_indices[name]
        telemetry[name]['ctrl'][i] = data.ctrl[idx]
        telemetry[name]['qpos'][i] = data.qpos[qpos_indices[name]]
        telemetry[name]['force'][i] = data.actuator_force[idx]
        
    # Avanzar al siguiente step:
    for _ in range(steps_per_frame):
        mujoco.mj_step(model, data)


Para entender que se evaluó, se generan unas gráficas mas extensivas, en que se muestra la posicion objetivo como linea punteada roja, la posicion real como linea azul, y cuanta fuerza hizo el motor en cada instante como linea verde.

In [ ]:
height_per_plot = 4.0
total_height = len(actuators_to_test) * height_per_plot

fig, axs = plt.subplots(len(actuators_to_test), 1, figsize=(15, total_height), sharex=True)

for ax, name in zip(axs, actuators_to_test):
    # Eje izquierdo: Posición
    ax.plot(time_log, telemetry[name]['ctrl'], 'r--', label='Target (rad/m)')
    ax.plot(time_log, telemetry[name]['qpos'], 'b-', label='Real (qpos)')
    ax.set_ylabel('Posición', color='blue')
    ax.tick_params(axis='y', labelcolor='blue')
    
    # Eje derecho: Fuerza/Torque
    ax2 = ax.twinx() 
    ax2.plot(time_log, telemetry[name]['force'], 'g-', alpha=0.4, label='Fuerza (N-m)')
    ax2.set_ylabel('Fuerza / Torque (N-m)', color='green')
    ax2.tick_params(axis='y', labelcolor='green')
    
    # Líneas de saturación (Opcional: dibuja el límite del forcerange)
    # Suponiendo que conoces el límite, por ejemplo 2.11
    # ax2.axhline(y=2.11, color='red', linestyle=':', alpha=0.3)
    # ax2.axhline(y=-2.11, color='red', linestyle=':', alpha=0.3)

    ax.set_title(f'Actuador: {name}', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Se buscaran las mejores calibraciones para cada actuador. Para esto, se hara un recorrido por diferentes valores y combinaciones posibles de factores de ganancia y de 'damping', observando que tanto logra la combinacion acercarse al valor objetivo, siendo 'ganadora' la combinacion que menor error medio cuadradro ('mean square error') tenga.

In [ ]:
# Identificadores
actuators_to_test = ["act_BaseHip","act_LeftShoulderArm", "act_Neck", "act_HipBody", "act_HeadBase", "act_HeadRot", "act_RightShoulderArm", "act_LeftForearm", "act_RightForearm", "act_LeftWrist", "act_RightWrist", "act_LeftLever_Slider", "act_RightLever_Slider"]
# act_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, act_name)
# joint_id = model.actuator_trnid[act_id, 0]
# dof_id = model.jnt_dofadr[joint_id]

# 2. ESPACIO DE BÚSQUEDA CONFIGURADO
kp_values = np.arange(100, 5000, 50)      # 400 a 6000, steps de 200
damping_values = np.arange(0, 155, 5)     # 35 a 150, steps de 5
gears = [1]                       # Mantengo gears para asegurar torque
target_val = {
    "act_BaseHip":          0.12,
    "act_HipBody":          0.40,
    "act_Neck":             0.50,
    "act_HeadBase":         0.50,
    "act_HeadRot":          0.50,
    "act_LeftShoulderArm":  0.07,   
    "act_RightShoulderArm": 0.20,
    "act_LeftForearm":      1.0,
    "act_RightForearm":     0.5,
    "act_LeftWrist":        1.0,
    "act_RightWrist":       1.0,
    "act_LeftLever_Slider": 0.007, 
    "act_RightLever_Slider":0.007,
}
sim_time = 1.5      
error_threshold = 0.07                     # Margen de tolerancia solicitado

best_configs = []
telemetry_data = {}

print(f"Iniciando optimización y captura de telemetría...")

for act_name in actuators_to_test:
    act_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, act_name)
    joint_id = model.actuator_trnid[act_id, 0]
    dof_id = model.jnt_dofadr[joint_id]
    qpos_adr = model.jnt_qposadr[joint_id]
    
    best_mse = float('inf')
    winner = None
    print(f'Buscando para {act_name = }')
    # --- FASE 1: GRID SEARCH ---
    for g in gears:
        for kp in kp_values:
            for d in damping_values:
                model.actuator_gear[act_id, 0] = g
                model.actuator_gainprm[act_id, 0] = kp
                model.actuator_biasprm[act_id, 1] = -kp
                model.dof_damping[dof_id] = d
                
                mujoco.mj_resetData(model, data)
                temp_qpos = []
                while data.time < sim_time:
                    data.ctrl[act_id] = target_val[act_name]
                    mujoco.mj_step(model, data)
                    temp_qpos.append(data.qpos[qpos_adr])
                
                mse = np.mean((np.array(temp_qpos) - target_val[act_name])**2)
                if abs(temp_qpos[-1] - target_val[act_name]) <= error_threshold:
                    if mse < best_mse:
                        best_mse = mse
                        winner = {'kp': kp, 'd': d, 'g': g}

    # --- FASE 2: CAPTURA DE TELEMETRÍA FINAL CON EL GANADOR ---
    if winner:
        # Aplicar el ganador
        model.actuator_gear[act_id, 0] = winner['g']
        model.actuator_gainprm[act_id, 0] = winner['kp']
        model.actuator_biasprm[act_id, 1] = -winner['kp']
        model.dof_damping[dof_id] = winner['d']
        
        mujoco.mj_resetData(model, data)
        t_log, p_log, f_log, c_log = [], [], [], []
        
        while data.time < sim_time:
            data.ctrl[act_id] = target_val[act_name]
            mujoco.mj_step(model, data)
            t_log.append(data.time)
            p_log.append(data.qpos[qpos_adr])
            f_log.append(data.actuator_force[act_id])
            c_log.append(data.ctrl[act_id])
            
        telemetry_data[act_name] = {'t': t_log, 'p': p_log, 'f': f_log, 'c': c_log, 'params': winner}
        best_configs.append({'Actuador': act_name, **winner, 'MSE': best_mse})
        print(f"✓ {act_name} listo.")
    else:
        print(f"✗ {act_name} sin solución.")

# --- 3. GENERACIÓN DE GRÁFICOS ---
n_plots = len(telemetry_data)
fig, axs = plt.subplots(n_plots, 1, figsize=(15, 4 * n_plots), sharex=True)
if n_plots == 1: axs = [axs]

for i, (name, d) in enumerate(telemetry_data.items()):
    ax = axs[i]
    # Posición (Eje Izquierdo)
    ax.plot(d['t'], d['c'], 'r--', linewidth=1.5, label='Target')
    ax.plot(d['t'], d['p'], 'b-', linewidth=1.2, label='Real')
    ax.set_ylabel('Posición (rad/m)', color='blue')
    ax.tick_params(axis='y', labelcolor='blue')
    
    # Fuerza (Eje Derecho)
    ax2 = ax.twinx()
    ax2.plot(d['t'], d['f'], 'g-', alpha=0.4, label='Fuerza')
    ax2.set_ylabel('Fuerza (N-m)', color='green')
    ax2.tick_params(axis='y', labelcolor='green')
    
    p = d['params']
    ax.set_title(f"Actuador: {name} | Gear:{p['g']} KP:{p['kp']} Damp:{p['d']}", fontweight='bold', loc='left')
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(loc='upper right')

plt.tight_layout()
plt.show()



In [ ]:
# Identificadores
act_name = "act_BaseHip"
act_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, act_name)
joint_id = model.actuator_trnid[act_id, 0]
dof_id = model.jnt_dofadr[joint_id]

# 2. ESPACIO DE BÚSQUEDA CONFIGURADO
kp_values = np.arange(1000, 5000, 500) 
damping_values = np.arange(1, 30, 2)
gears = [1] 
sim_time = 1.0 
error_threshold = 0.02

target_value = 0.15 
valid_results = []

print(f"Iniciando Grid Search exhaustivo...")
total_mass = np.sum(model.body_mass)
print(f"Masa total del robot: {total_mass:.2f} kg")
for g in gears:
    for kp in kp_values:
        for d in damping_values:
            # --- APLICAR PARÁMETROS DINÁMICAMENTE ---
            model.actuator_gear[act_id, 0] = g
            model.actuator_gainprm[act_id, 0] = kp
            model.actuator_biasprm[act_id, 1] = -kp
            model.dof_damping[dof_id] = d
            
            mujoco.mj_resetData(model, data)
            temp_qpos = []
            temp_time = []
            
            while data.time < sim_time:
                data.ctrl[act_id] = target_value
                mujoco.mj_step(model, data)
                temp_qpos.append(data.qpos[model.jnt_qposadr[joint_id]])
                temp_time.append(data.time)
            
            # --- FILTRO DE PRECISIÓN ---
            final_pos = temp_qpos[-1]
            if abs(final_pos - target_value) <= error_threshold:
                # Calculamos el error cuadrático medio para ordenarlos por calidad
                mse = np.mean((np.array(temp_qpos) - target_value)**2)
                valid_results.append({
                    'kp': kp, 'd': d, 'gear': g, 
                    'mse': mse, 'qpos': temp_qpos, 'time': temp_time,
                    'final_val': final_pos
                })
            print(f"probando: {g = }, {kp = }, {d = }, {abs(final_pos - target_value) = }")
                

print(f"Búsqueda finalizada. Se encontraron {len(valid_results)} configuraciones válidas.")

# 3. VISUALIZACIÓN FILTRADA
if len(valid_results) > 0:
    # Ordenamos por menor error (el que mejor sigue la curva)
    valid_results.sort(key=lambda x: x['mse'])
    
    # Limitamos a mostrar los mejores 12 para no saturar el Notebook
    display_count = min(len(valid_results), 12)
    rows = int(np.ceil(display_count / 2))
    fig, axs = plt.subplots(rows, 2, figsize=(15, rows * 4))
    fig.suptitle(f'Configuraciones que cumplen Error < {error_threshold} rad\n', fontsize=16)
    axs = axs.flatten()

    for i in range(display_count):
        res = valid_results[i]
        axs[i].plot(res['time'], [target_value]*len(res['time']), 'r--', label='Target')
        axs[i].plot(res['time'], res['qpos'], 'b-', label='Posición Real')
        axs[i].set_title(f"Gear:{res['gear']} | KP:{res['kp']} | Damp:{res['d']}\nFinal Pos: {res['final_val']:.4f}")
        axs[i].legend(loc='lower right')
        axs[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    
    best = valid_results[0]
    print(f"--- MEJOR CONFIGURACIÓN ENCONTRADA ---")
    print(f"Gear: {best['gear']} | KP: {best['kp']} | Damping: {best['d']} | mse: {best['mse']}")
    # Gear: 1 | KP: 400 | Damping: 35
else:
    print("Ninguna combinación logró entrar en el margen de 0.02 rad. Prueba aumentando el Gear.")
    # --- 4. TABLA RESUMEN ---
print("\n" + "="*50)
import pandas as pd
print(pd.DataFrame(best_configs).to_string(index=False))
print("="*50)

Como los resultados no son los esperados, se van a hacer pruebas con cambios en el archivo original. Por ejemplo, se agrega el termino 'kv' que es el componente derivativo para elcontrol PIDde losmotores

In [ ]:
import multiprocessing as mp
import mujoco
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from functools import partial

XML_PATH = "Cuerpo\DUM4.xml"  

def search_one_actuator(act_name, kp_values, kd_values, damping_values, gears,
                        target_dict, sim_time, error_threshold):
    """Worker independiente: carga su propia copia del modelo."""
    # Carga silenciosa del modelo
    model = mujoco.MjModel.from_xml_path(XML_PATH)
    data  = mujoco.MjData(model)

    act_id   = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, act_name)
    joint_id = model.actuator_trnid[act_id, 0]
    dof_id   = model.jnt_dofadr[joint_id]
    qpos_adr = model.jnt_qposadr[joint_id]
    
    # El valor objetivo específico para este actuador
    target = target_dict[act_name]

    best_mse = float('inf')
    winner   = None
    
    # Flush=True asegura que el print salga del proceso hijo a la terminal principal
    print(f"[START] {act_name}", flush=True)

    for g in gears:
        for kp in kp_values:
            for kd in kd_values:
                for d in damping_values:
                    print(f'{act_name} prueba {d}, {kd}, {kp}, {g}', flush=True)
                    model.actuator_gear[act_id, 0]    = g
                    model.actuator_gainprm[act_id, 0] = kp
                    model.actuator_biasprm[act_id, 1] = -kp
                    model.actuator_biasprm[act_id, 2] = -kd
                    model.dof_damping[dof_id]         = d

                    mujoco.mj_resetData(model, data)
                    temp_qpos = []
                    
                    while data.time < sim_time:
                        data.ctrl[act_id] = target
                        mujoco.mj_step(model, data)
                        temp_qpos.append(data.qpos[qpos_adr])

                    final_pos = temp_qpos[-1]
                    mse = np.mean((np.array(temp_qpos) - target)**2)
                    
                    if abs(final_pos - target) <= error_threshold:
                        print(f'{act_name} tiene al menos 1 config posible')
                        if mse < best_mse:
                            best_mse = mse
                            winner = {'kp': kp, 'kd': kd, 'd': d, 'g': g, 'final_val': final_pos}

    # Telemetría final con el ganador para retornar datos de gráfica
    if winner:
        model.actuator_gear[act_id, 0]    = winner['g']
        model.actuator_gainprm[act_id, 0] = winner['kp']
        model.actuator_biasprm[act_id, 1] = -winner['kp']
        model.actuator_biasprm[act_id, 2] = -winner['kd']
        model.dof_damping[dof_id]         = winner['d']

        mujoco.mj_resetData(model, data)
        t_log, p_log, f_log = [], [], []
        while data.time < sim_time:
            data.ctrl[act_id] = target
            mujoco.mj_step(model, data)
            t_log.append(data.time)
            p_log.append(data.qpos[qpos_adr])
            f_log.append(data.actuator_force[act_id])

        print(f"[SUCCESS] {act_name} optimizado.", flush=True)
        return {
            'act_name': act_name, 'winner': winner, 'mse': best_mse,
            't': t_log, 'p': p_log, 'f': f_log, 'target': target
        }
    else:
        print(f"[FAILED] {act_name} sin solución.", flush=True)
        return {'act_name': act_name, 'winner': None}

if __name__ == "__main__":
    actuators_to_test = ["act_BaseHip", "act_LeftShoulderArm", "act_Neck", "act_HipBody",
                         "act_HeadBase", "act_HeadRot", "act_RightShoulderArm",
                         "act_LeftForearm", "act_RightForearm", "act_LeftWrist",
                         "act_RightWrist", "act_LeftLever_Slider", "act_RightLever_Slider"]

    # Espacio de búsqueda
    kp_values      = np.arange(100, 2000, 200)
    kd_values      = np.arange(0, 200, 20)
    damping_values = np.arange(0, 155, 10)
    gears          = [1, 5, 10]

    target_val = {
        "act_BaseHip": 0.12, "act_HipBody": 0.40, "act_Neck": 0.50, 
        "act_HeadBase": 0.50, "act_HeadRot": 0.50, "act_LeftShoulderArm": 0.07,
        "act_RightShoulderArm": 0.20, "act_LeftForearm": 1.0, "act_RightForearm": 0.5,
        "act_LeftWrist": 1.0, "act_RightWrist": 1.0, "act_LeftLever_Slider": 0.018,
        "act_RightLever_Slider": 0.018,
    }
    
    sim_time = 1.2
    error_threshold = 0.03

    N_WORKERS = max(1, mp.cpu_count() - 1)
    
    worker_func = partial(
        search_one_actuator,
        kp_values=kp_values, kd_values=kd_values, damping_values=damping_values,
        gears=gears, target_dict=target_val, sim_time=sim_time, error_threshold=error_threshold
    )

    with mp.Pool(N_WORKERS) as pool:
        results = pool.map(worker_func, actuators_to_test)

    # Filtrar solo los exitosos para graficar
    valid_results = [r for r in results if r['winner'] is not None]
    valid_results.sort(key=lambda x: x['mse'])

    if valid_results:
        display_count = len(valid_results)
        cols = 2
        rows = int(np.ceil(display_count / cols))
        fig, axs = plt.subplots(rows, cols, figsize=(15, rows * 4))
        axs = axs.flatten() if display_count > 1 else [axs]

        for i, res in enumerate(valid_results):
            ax = axs[i]
            w = res['winner']
            ax.plot(res['t'], [res['target']]*len(res['t']), 'r--', label='Target')
            ax.plot(res['t'], res['p'], 'b-', label='Posición')
            
            # Eje secundario para fuerza
            ax2 = ax.twinx()
            ax2.plot(res['t'], res['f'], 'g-', alpha=0.3, label='Fuerza')
            
            ax.set_title(f"{res['act_name']}\nGear:{w['g']} | KP:{w['kp']} | KD:{w['kd']} | D:{w['d']}")
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Tabla Resumen
        summary = []
        for r in valid_results:
            summary.append({'Actuador': r['act_name'], **r['winner'], 'MSE': r['mse']})
        print("\n" + "="*60)
        print(pd.DataFrame(summary).to_string(index=False))
        print("="*60)
    else:
        print("No se encontraron configuraciones válidas.")